In [1]:
import pandas as pd # Pandas kütüphanesinin fonksiyonlarına pd alias'ı ile ulaşacağız.
import matplotlib.pyplot as plt # veri görselleştirme alanında kullanacağımız kütüphane
import seaborn as sns # Matplotlib'in üzerine kurulan daha az kodla yazılan veri görselleştirme kütüphanesi

In [2]:
df = pd.read_excel("../data/online_retail_II.xlsx") # Excel verisini DataFrame formatına dönüştürme

In [3]:
df.info() # veri setinin sütunları, kapladığı yer bunları öğrenelim.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 525461 entries, 0 to 525460
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      525461 non-null  object        
 1   StockCode    525461 non-null  object        
 2   Description  522533 non-null  object        
 3   Quantity     525461 non-null  int64         
 4   InvoiceDate  525461 non-null  datetime64[ns]
 5   Price        525461 non-null  float64       
 6   Customer ID  417534 non-null  float64       
 7   Country      525461 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 32.1+ MB


In [4]:
print(f"Veri setindeki gözlem sayısı: {df.shape[0]}") # veri setinin satır sayısı
print(f"Veri setindeki sütun sayısı: {df.shape[1]}") # veri setinin sütun sayısı

Veri setindeki gözlem sayısı: 525461
Veri setindeki sütun sayısı: 8


In [5]:
df.head() # veri setinin ilk 5 satırına göz atalım. Head fonksiyonuna değer vermezsek default olarak ilk 5 satırı gösterir.

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [6]:
# veri setindeki sayısal sütunlar üzerinde mod medyan min max gibi temel istatistik değerlerine bakalım.
df.describe() # yukarıdaki işlem için describe fonksiyonunu kullanıyoruz.

,Quantity,InvoiceDate,Price,Customer ID
count,525461.000000,525461,525461.000000,417534.000000
mean,10.337667,2010-06-28 11:37:36.845017856,4.688834,15360.645478
min,-9600.000000,2009-12-01 07:45:00,-53594.360000,12346.000000
25%,1.000000,2010-03-21 12:20:00,1.250000,13983.000000
50%,3.000000,2010-07-06 09:51:00,2.100000,15311.000000
75%,10.000000,2010-10-15 12:45:00,4.210000,16799.000000
max,19152.000000,2010-12-09 20:01:00,25111.090000,18287.000000
std,107.424110,NaN,146.126914,1680.811316


In [7]:
# son olarak veri setindeki sütunların her birinde boş değer olup olmadığını kontrol edelim. Kuracağımız modellerde boş veri olmaması gerekir. İleride boş verileri nasıl dolduracağımıza bakacağız.
df.isna().sum()

Invoice             0
StockCode           0
Description      2928
Quantity            0
InvoiceDate         0
Price               0
Customer ID    107927
Country             0
dtype: int64

### EDA Sonuç
Veri seti > 525.461 satır ve 8 sütundan oluşuyor.
* Invoice > Fatura numarası. "C" ile başlıyorsa — iptal siparişi
* StockCode > Ürün kodu
* Description > Ürün adı
* Quantity > Satılan adet. Negatifse — iade
* Invoice > DateFatura tarihi ve saati
* Price > Birim fiyat (£)
* Customer ID > Müşteri numarası. Boşsa — anonim müşteri
* Country > Müşterinin bulunduğu ülke

2928 siparişin ürün adı bilgisi eksik. Bunları bilinmiyor olarak dolduracağız.
107927 siparişin hangi müşteri tarafından alındığına dair bir bilgi yok. Bu sütundaki boş verilere ne yapacağımıza ileride göz atacağız.

### Veri Temizleme

In [8]:
df["Description"] = df["Description"].fillna(value = "Unknown") # ürün adı bilinmeyen siparişlerdeki ürün adlarını Unknown (bilinmiyor) olarak dolduralım.
df = df.dropna(subset=["Customer ID"])

In [9]:
# verilerin dolup dolmadığını kontrol edelim.
df[["Description", "Customer ID"]].isna().sum()

Description    0
Customer ID    0
dtype: int64

In [10]:
df["Customer ID"].dtype # customer id sütunu tam sayı olduğu için bunu float tipinden integer'a çevireceğiz ancak bu çeviriyi yapmadan önce son defa bu sütunun veri tipini kontrol edelim.

dtype('float64')

In [11]:
df.loc[:, "Customer ID"] = df["Customer ID"].astype(int)

In [12]:
df = df[~df["Invoice"].astype(str).str.startswith("C")] # iptal olmayan siparişleri tutuyoruz. ~ yazdığımız koşulun tersine uyan satırları bize veriyor.
df = df[df["Quantity"] > 0] # negatif satış adetleri (iade)'leri veri setinden kaldırıyoruz.
df = df[df["Price"] > 0] # ücret bilgisi 0 veya negatif olamayacağı için price > 0 olan satırları tutuyoruz.

In [13]:
# güncel veri setimizin gözlem sayısına bakalım.
print(df.shape[0])

407664


In [14]:
# tekrar eden satır var mı kontrol edelim.
df.duplicated().sum()

np.int64(6748)

In [15]:
# 6748 tane tekrar eden satırı veri setinden kaldıralım.
df = df.drop_duplicates()
print(df.shape[0]) # yeniden veri setindeki gözlem sayısına bakalım.

400916


### Feature Engineering

In [16]:
df["Revenue"] = df["Quantity"] * df["Price"] # her bir müşterinin sipariş başına harcadığı toplam parayı yeni bir sütun olarak türetiyoruz.

### RFM 
Recency > Müşterinin en son alışveriş tarihinden bu yana kaç gün geçtiğine bakar. Bunun için bir referans tarihi gerekir bu tarih genellikle veri setindeki en son gün + 1 gün olarak hesaplanır.
Frequency > Müşterinin kaç kere sipariş yaptığı bilgisini verir.
Monetary > Müşterinin toplam harcamasını verir. 

In [18]:
# RFM hesaplama
referans_tarihi = df["InvoiceDate"].max() + pd.Timedelta(days=1)# referans günü tespiti
print(f"Referans tarihi: {referans_tarihi}")

Referans tarihi: 2010-12-10 20:01:00


In [20]:
type(referans_tarihi)

pandas._libs.tslibs.timestamps.Timestamp

In [19]:
rfm = df.groupby("Customer ID").agg({"InvoiceDate": lambda x: (referans_tarihi - x.max()).days}).reset_index()
rfm.head()


,Customer ID,InvoiceDate
0,12346.0,165
1,12347.0,3
2,12348.0,74
3,12349.0,43
4,12351.0,11


In [ ]:
# frequency değeri hesaplayalım.
rfm["Frequency"] = df.groupby("Customer ID").agg({"Invoice":"count"})

In [32]:
# monetary değeri hesaplayalım.
rfm["Monetary"] = df.groupby("Customer ID").agg({"Revenue":"sum"})
rfm.head() # RFM dataframe'ini kontrol edelim.

,Recency,Frequency,Monetary
Customer ID,,,
12346.0,165,33,372.86
12347.0,3,71,1323.32
12348.0,74,20,222.16
12349.0,43,102,2671.14
12351.0,11,21,300.93


In [33]:
# şimdi de Customer ID 'leri indeks olmaktan çıkarıp integer türünde bir sütuna dönüştürüyoruz. Amacımız filtreleme, görselleştirme gibi işlemlerin daha kolay yapılabilmesini sağlamak.
rfm = rfm.reset_index()
rfm["Customer ID"] = rfm["Customer ID"].astype(int)
rfm.head()

,Customer ID,Recency,Frequency,Monetary
0,12346,165,33,372.86
1,12347,3,71,1323.32
2,12348,74,20,222.16
3,12349,43,102,2671.14
4,12351,11,21,300.93


In [34]:
rfm["R_Score"] = pd.qcut(rfm["Recency"], q=4, labels=[4, 3, 2, 1]) # en düşük Recency değerine sahip olan müşteri en güncel olduğu için ona 4 veriyoruz.
rfm["F_Score"] = pd.qcut(rfm["Frequency"].rank(method="first"), q=4, labels=[1, 2, 3, 4]) # en yüksek Frequency değerine sahip olan müşteri en fazla sipariş yaptığı için ona 4 veriyoruz.
rfm["M_Score"] = pd.qcut(rfm["Monetary"].rank(method="first"), q=4, labels=[1, 2, 3, 4]) # en yüksek Monetary değerine sahip olan müşteri şirkete en çok parayı kazandırdığı için ona 4 veriyoruz.
# rank(method="first") ifadesi aynı değere sahip verilerin farklı skorlar olmasını sağlar. Aksi takdirde qcut fonksiyonu veriyi bölemez.
rfm["RFM_Score"] = rfm["R_Score"].astype(int) + rfm["F_Score"].astype(int) + rfm["M_Score"].astype(int) # her bir müşteri için genel RFM skorunu hesaplayalım.
rfm.head()


,Customer ID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score
0,12346,165,33,372.86,1,2,2,5
1,12347,3,71,1323.32,4,3,3,10
2,12348,74,20,222.16,2,2,1,5
3,12349,43,102,2671.14,3,4,4,11
4,12351,11,21,300.93,4,2,1,7


In [35]:
rfm["Segment"] = pd.cut(  # müşterileri RFM skorlarına göre gruplarına ayırıp Segment etiketi atayalım.
    rfm["RFM_Score"],  
    bins=[0, 3, 6, 9, 12],
    labels=["Kayıp Müşteriler", "Risk Altındakiler", "Sadık Müşteriler", "Şampiyonlar"]
)
rfm["Segment"].value_counts() 

Segment
Sadık Müşteriler     1383
Risk Altındakiler    1317
Şampiyonlar          1234
Kayıp Müşteriler      378
Name: count, dtype: int64

### Görselleştirme
3 grafik yapacağız.

Grafikler                                                
                                                           
1) Segment dağılımı (bar chart)                            
2) Segment bazında ortalama Monetary                       
3) Recency vs Monetary scatter plot                        

In [36]:
import plotly.express as px # interaktif grafikler yapacağımız için veri görselleştirme aracı olarak plotly kütüphanesini kullanacağız.

In [54]:
segment_counts = rfm["Segment"].value_counts().reset_index()
segment_counts.columns = ["Segment", "Count"] 
segment_counts.head()

<class 'pandas.core.frame.DataFrame'>


In [56]:
fig = px.bar(segment_counts, x="Segment", y="Count", title="Segment Counts")
fig.show()

In [59]:
## Segment bazında ortalama monetary hesaplayalım.
avg_monetary_per_segments = rfm.groupby("Segment").agg({"Monetary":"mean"}).reset_index()
avg_monetary_per_segments.head()

C:\Users\asus\AppData\Local\Temp\ipykernel_14220\3449018701.py:2: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



,Segment,Monetary
0,Kayıp Müşteriler,154.394048
1,Risk Altındakiler,403.540814
2,Sadık Müşteriler,1179.916488
3,Şampiyonlar,5329.485445


In [62]:
fig = px.bar(
    segment_counts,
    x="Segment",
    y="Count",
    title="Müşteri Segment Dağılımı",
    labels={"Segment": "Segment", "Count": "Müşteri Sayısı"},
    color="Segment"
)
fig.show()

In [64]:
scatter = px.scatter(
    rfm,
    x="Recency",
    y="Monetary",
    color="Segment",
    title="Recency vs Monetary — Segment Bazında",
    labels={"Recency": "Recency (Gün)", "Monetary": "Toplam Harcama (£)"}
)
scatter.show()

### Scatter Grafiğinin yorumu
Şampiyonlar genelde sol altta yoğunlaşmış yani bu grup şirketten sürekli alışveriş yapıyor ama çok para harcamıyor. Sadık müşteriler de şirketten sürekli alışveriş yapıyor ancak şampiyonlara göre daha az para harcıyor. Risk altındakiler bir süredir alışveriş yapmıyor. Kayıp müşteriler ise uzun süredir alışveriş yapmıyor.